# Python reference 10: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-10-errors-logging-and-testing.ipynb)

The exception hierarchy, `try`/`except`/`else`/`finally`, chaining, custom hierarchies, notes and exception groups; context managers and `contextlib`; logging with levels, handlers and JSON output; pytest and `unittest.mock`; and modules, the `__main__` guard and `argparse`. Practise it in [Python practice 10: errors, logging and testing](../practice/python-practice-10-errors-logging-and-testing.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Exceptions](#exceptions) | [The exception hierarchy](#the-exception-hierarchy), [`try`, `except`, `else`, `finally`](#try-except-else-finally), [Raising, chaining and custom exceptions](#raising-chaining-and-custom-exceptions), [Exception groups and `except*`](#exception-groups-and-except) |
| [Context managers](#context-managers) | [`with`, `__enter__` and `__exit__`](#with-__enter__-and-__exit__), [`contextlib`](#contextlib) |
| [Logging](#logging) | [Loggers, levels and handlers](#loggers-levels-and-handlers), [Structured (JSON) logs](#structured-json-logs) |
| [Testing](#testing) | [pytest essentials](#pytest-essentials), [Test doubles with `unittest.mock`](#test-doubles-with-unittestmock) |
| [Modules and scripts](#modules-and-scripts) | [Imports, packages and the `__main__` guard](#imports-packages-and-the-__main__-guard) |

## Exceptions

### The exception hierarchy

| class | raised for |
| --- | --- |
| `BaseException` | root; also parent of `KeyboardInterrupt`, `SystemExit`, `GeneratorExit` (do not catch these by accident) |
| `Exception` | root of everything you normally handle |
| `ValueError` / `TypeError` | right type, bad value / wrong type |
| `KeyError` / `IndexError` (both `LookupError`) | missing key / index out of range |
| `AttributeError`, `NameError` | missing attribute / undefined name |
| `OSError` (`FileNotFoundError`, `PermissionError`, `TimeoutError`, `ConnectionError`) | operating-system and I/O failures |
| `ArithmeticError` (`ZeroDivisionError`, `OverflowError`) | arithmetic |
| `RuntimeError`, `NotImplementedError` | generic failures / abstract methods |
| `json.JSONDecodeError` (a `ValueError`), `UnicodeDecodeError` (a `ValueError`) | parsing |

**Notes:** catch the **narrowest** class you can handle. A bare `except:` or `except BaseException` also swallows
Ctrl-C and `sys.exit()`. `except Exception` is acceptable only at a top-level boundary that logs and re-raises or
records the failure.

In [ ]:
import json

for bad in ['{"a": 1', b"\xff".decode("latin-1").encode("utf-8")]:
    try:
        json.loads(bad) if isinstance(bad, str) else bad.decode("ascii")
    except ValueError as e:
        print(type(e).__name__, "is a ValueError:", isinstance(e, ValueError), [c.__name__ for c in type(e).__mro__[:3]])
print(issubclass(FileNotFoundError, OSError), issubclass(KeyError, LookupError), issubclass(KeyboardInterrupt, Exception))

### `try`, `except`, `else`, `finally`

```python
try:
    ...                      # only the code that can fail
except (KeyError, ValueError) as error:
    ...                      # handle these types
else:
    ...                      # runs only if nothing was raised
finally:
    ...                      # always runs: cleanup
```

**Notes:** keep the `try` block small so you do not catch errors from unrelated lines; put the follow-up in `else`.
`finally` runs even on `return` or an unhandled exception. The `as error` name is deleted after the block.

In [ ]:
def parse_amount(text):
    try:
        value = float(text)
    except (TypeError, ValueError):
        return None
    else:
        return round(value, 2)
    finally:
        print(f"  parsed {text!r}")

print([parse_amount(x) for x in ["12.345", "abc", None]])

### Raising, chaining and custom exceptions

| syntax | effect |
| --- | --- |
| `raise ValueError("msg")` | raise a new exception |
| `raise` | re-raise the current exception unchanged (inside `except`) |
| `raise NewError("msg") from error` | chain: `__cause__` is `error`; the traceback shows both |
| `raise NewError("msg") from None` | hide the original context |
| `error.add_note("context")` (3.11+) | attach extra lines shown in the traceback |

Define a small **hierarchy** for your library or pipeline: a base class callers can catch, and specific subclasses
that carry structured attributes (the field, the record id) rather than only a message.

In [ ]:
class PipelineError(Exception):
    """Base class for pipeline failures."""

class ValidationError(PipelineError):
    def __init__(self, field, message):
        super().__init__(f"{field}: {message}")
        self.field = field

class SourceError(PipelineError):
    pass

def read_config(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as error:
        raise SourceError("config is not valid JSON") from error

try:
    read_config("{bad")
except PipelineError as e:
    print(type(e).__name__, "|", e, "| caused by", type(e.__cause__).__name__)

try:
    raise ValidationError("amount", "must be positive")
except ValidationError as e:
    e.add_note("record id=7, file=orders.csv")
    print(e.field, e, e.__notes__)

### Exception groups and `except*`

`ExceptionGroup(message, [errors])` (3.11+) raises several unrelated errors together, as `asyncio.TaskGroup` does when
several tasks fail. `except* SomeError` handles the matching members of a group and lets the rest propagate.

In [ ]:
def load_all(sources):
    errors, loaded = [], []
    for s in sources:
        try:
            if s.startswith("bad"):
                raise ValueError(f"cannot parse {s}")
            if s.startswith("gone"):
                raise FileNotFoundError(s)
            loaded.append(s)
        except Exception as e:
            errors.append(e)
    if errors:
        raise ExceptionGroup(f"{len(errors)} sources failed", errors)
    return loaded

try:
    load_all(["a", "bad1", "gone2", "bad3"])
except* ValueError as group:
    print("parse failures:", [str(e) for e in group.exceptions])
except* FileNotFoundError as group:
    print("missing:", [str(e) for e in group.exceptions])

## Context managers

### `with`, `__enter__` and `__exit__`

`with manager as value:` calls `manager.__enter__()` (its result is bound to `value`) and guarantees
`manager.__exit__(exc_type, exc, tb)` on the way out, whether the block finished or raised. Returning a true value from
`__exit__` **suppresses** the exception. Files, locks, database connections and transactions are context managers.

In [ ]:
class Transaction:
    def __init__(self, store):
        self.store = store
    def __enter__(self):
        self.snapshot = dict(self.store)
        return self.store
    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None:
            self.store.clear()
            self.store.update(self.snapshot)
            print(f"  rolled back after {exc_type.__name__}")
        return False   # do not suppress

balances = {"a": 100, "b": 0}
try:
    with Transaction(balances) as s:
        s["a"] -= 30
        s["b"] += 30
        raise ConnectionError("network dropped")
except ConnectionError:
    pass
print(balances)

### `contextlib`

| tool | use |
| --- | --- |
| `@contextmanager` | write a context manager as a generator: code before `yield` is enter, after is exit (wrap the `yield` in `try/finally`) |
| `suppress(*exceptions)` | ignore specific exceptions: `with suppress(FileNotFoundError): path.unlink()` |
| `ExitStack()` | enter a variable number of context managers (open N files) and close them all |
| `closing(obj)` | call `obj.close()` on exit |
| `nullcontext(value)` | a do-nothing stand-in when a manager is optional |
| `redirect_stdout(buffer)` | capture prints |

In [ ]:
import os
import tempfile
from contextlib import ExitStack, contextmanager, redirect_stdout, suppress
from io import StringIO
from pathlib import Path

@contextmanager
def env_vars(**values):
    old = {k: os.environ.get(k) for k in values}
    os.environ.update(values)
    try:
        yield
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

with env_vars(STAGE="test"):
    print("inside:", os.environ["STAGE"])
print("after:", os.environ.get("STAGE"))

with suppress(FileNotFoundError):
    Path("/definitely/missing.txt").unlink()

root = Path(tempfile.mkdtemp())
with ExitStack() as stack:
    files = [stack.enter_context(open(root / f"part-{i}.txt", "w", encoding="utf-8")) for i in range(3)]
    for i, f in enumerate(files):
        f.write(f"part {i}")
print(all(f.closed for f in files), sorted(p.name for p in root.iterdir()))

buffer = StringIO()
with redirect_stdout(buffer):
    print("captured")
print(repr(buffer.getvalue()))

## Logging

### Loggers, levels and handlers

| piece | role |
| --- | --- |
| `logger = logging.getLogger(__name__)` | one logger per module, named by the module path, so output can be filtered by package |
| levels | `DEBUG` < `INFO` < `WARNING` < `ERROR` < `CRITICAL`; a logger and each handler drop records below their level |
| handlers | where records go: `StreamHandler` (stderr), `FileHandler`, `RotatingFileHandler`, ... |
| formatters | how records look: `"%(asctime)s %(levelname)s %(name)s %(message)s"` |
| `logging.basicConfig(level=..., format=...)` | configure the root logger once, in the application entry point (not in libraries) |

| call | notes |
| --- | --- |
| `logger.info("loaded %d rows from %s", n, path)` | pass arguments rather than an f-string: formatting is skipped when the level is disabled |
| `logger.exception("load failed")` | inside `except`: logs at ERROR with the traceback |
| `logger.info("msg", extra={"run_id": r})` | extra fields on the record, for structured (JSON) formatters |

**Notes:** libraries should only create loggers, never configure handlers. Never log secrets or full records with
personal data.

In [ ]:
import logging
import sys

logger = logging.getLogger("pipeline.orders")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-7s %(name)s run=%(run_id)s %(message)s"))
logger.handlers, logger.propagate = [handler], False
logger.setLevel(logging.INFO)

ctx = {"run_id": "r-42"}
logger.debug("not shown: below INFO", extra=ctx)
logger.info("loaded %d rows from %s", 1180, "orders.csv", extra=ctx)
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("division failed", extra=ctx)

### Structured (JSON) logs

Log aggregators (CloudWatch, Datadog, ELK) search JSON fields far better than free text. A small `Formatter`
subclass turns each record into one JSON line, including any `extra` fields.

In [ ]:
import json

STANDARD = set(vars(logging.makeLogRecord({})))

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {"level": record.levelname, "logger": record.name, "message": record.getMessage()}
        payload.update({k: v for k, v in vars(record).items() if k not in STANDARD})
        if record.exc_info:
            payload["error"] = self.formatException(record.exc_info).splitlines()[-1]
        return json.dumps(payload, default=str)

handler.setFormatter(JsonFormatter())
logger.info("batch done", extra={"run_id": "r-42", "rows": 1180, "seconds": 3.2})

## Testing

### pytest essentials

pytest discovers `test_*.py` files and `test_*` functions and rewrites plain `assert` statements to show both sides on
failure.

```python
import pytest

def test_normalise_strips_and_lowers():
    assert normalise("  A ") == "a"

@pytest.mark.parametrize("raw, expected", [("1", 1), ("1.5", 1.5), ("", None)])
def test_parse(raw, expected):
    assert parse(raw) == expected

def test_rejects_negative():
    with pytest.raises(ValueError, match="positive"):
        validate(-1)

@pytest.fixture
def orders():
    return [{"id": 1, "amount": 120.0}]

def test_total(orders, tmp_path, monkeypatch):
    monkeypatch.setenv("STAGE", "test")        # undone after the test
    out = tmp_path / "out.csv"                 # a fresh directory per test
    assert total(orders) == 120.0
```

| tool | use |
| --- | --- |
| fixtures (`@pytest.fixture`) | shared setup injected by parameter name; `yield` in a fixture for teardown; `scope="session"` for expensive ones (a SparkSession) |
| `parametrize` | one test, many input rows |
| `pytest.raises`, `pytest.approx` | exceptions; float comparisons |
| `tmp_path`, `monkeypatch`, `caplog`, `capsys` | built-in fixtures for files, environment, logs and output |

**Notes:** test the pure transformation functions directly; keep I/O at the edges so most tests need no files or
network. `assert` statements are removed under `python -O`, so never use `assert` for runtime validation in
production code.

In [ ]:
import math

def normalise(s):
    return s.strip().lower()

cases = [("  A ", "a"), ("b", "b"), ("\tC\n", "c")]
# What a parametrised test checks, shown as output: (input, actual, expected, passed)
print([(raw, normalise(raw), expected, normalise(raw) == expected) for raw, expected in cases])
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "| math.isclose:", math.isclose(0.1 + 0.2, 0.3), "(pytest.approx does the same)")

### Test doubles with `unittest.mock`

| tool | use |
| --- | --- |
| `Mock()` / `MagicMock()` | an object that accepts any call and records it |
| `m.return_value = x` / `m.side_effect = exc_or_list_or_func` | canned results, raised errors, a sequence of results |
| `m.assert_called_once_with(...)`, `m.call_count`, `m.call_args_list` | verify interactions |
| `patch("module.name", ...)` | replace a name **where it is looked up** for the duration of a `with` block or test |
| `Mock(spec=RealClass)` | reject attributes the real class does not have |

**Notes:** prefer passing dependencies in (a `client` argument, a `sleep` function) over patching module globals;
the code becomes easier to test without `patch` at all.

In [ ]:
import time
from unittest.mock import Mock, patch

def fetch_with_retry(client, key, attempts=3, sleep=time.sleep):
    for attempt in range(1, attempts + 1):
        try:
            return client.get(key)
        except ConnectionError:
            if attempt == attempts:
                raise
            sleep(2 ** attempt)

client = Mock()
client.get.side_effect = [ConnectionError("blip"), ConnectionError("blip"), {"id": 7}]
fake_sleep = Mock()
print(fetch_with_retry(client, "orders/7", sleep=fake_sleep), client.get.call_count, fake_sleep.call_args_list)

with patch("time.time", return_value=1_800_000_000.0):
    print(time.time())

## Modules and scripts

### Imports, packages and the `__main__` guard

| concept | notes |
| --- | --- |
| module | a `.py` file; imported once per process and cached in `sys.modules` |
| package | a directory of modules (with `__init__.py`); import as `package.module` |
| `from package import name` / `from . import sibling` | absolute imports are clearer; relative imports work only inside packages |
| `if __name__ == "__main__":` | code that runs when the file is executed as a script, not when imported |
| `python -m package.module` | run a module as a script with the package on the path |
| `sys.exit(code)` | exit status: 0 success, non-zero failure (schedulers read it) |

`argparse` builds command-line interfaces with types, defaults, help text and errors (which exit with status 2).

In [ ]:
import argparse
from datetime import date

def build_parser():
    parser = argparse.ArgumentParser(prog="load_orders", description="Load one day of orders.")
    parser.add_argument("--date", type=date.fromisoformat, required=True, help="partition date, YYYY-MM-DD")
    parser.add_argument("--limit", type=int, default=1000)
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--table", choices=["orders", "refunds"], default="orders")
    return parser

def main(argv=None):
    args = build_parser().parse_args(argv)
    print(f"  loading {args.table} for {args.date} limit={args.limit} dry_run={args.dry_run}")
    return 0

print("exit code", main(["--date", "2026-11-27", "--dry-run"]))
try:
    main(["--date", "27/11/2026"])
except SystemExit as e:
    print("argparse exited with", e.code)

if __name__ == "__main__":
    pass  # in a script: raise SystemExit(main())

**Next:** practise these entries in [Python practice 10: errors, logging and testing](../practice/python-practice-10-errors-logging-and-testing.ipynb).